\\
🔹 STEP 1: Install dependencies


In [ ]:
!pip install transformers torch finnhub-python

🔹 STEP 2: Setup + Load Models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import finnhub
import os
from datetime import datetime, timedelta

# 🔐 Set your Finnhub API key
os.environ["FINNHUB_API_KEY"] = "d7dcpdhr01qggoensdf0d7dcpdhr01qggoensdfg"

# Initialize client
client = finnhub.Client(api_key=os.environ["FINNHUB_API_KEY"])

# Load FinBERT model
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

model.eval()  # set model to evaluation mode

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

🔹 STEP 3: Main Sentiment Function

In [ ]:
def get_sentiment_score(symbol):
    """
    Returns sentiment score between -1 and 1
    based on last 15 days news
    """

    try:
        # Get date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=15)

        # Fetch news
        news = client.company_news(
            symbol,
            _from=start_date.strftime('%Y-%m-%d'),
            to=end_date.strftime('%Y-%m-%d')
        )

        # No news → neutral
        if not news:
            return 0.0

        texts = []

        # Limit to avoid slow processing
        for article in news[:50]:
            headline = article.get('headline', '')
            summary = article.get('summary', '')

            text = (headline + " " + summary).strip()

            if text:  # avoid empty text
                texts.append(text)

        # If no valid text
        if len(texts) == 0:
            return 0.0

        # Tokenize
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Model inference
        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=1)

        # Convert to sentiment scores
        scores = []
        for p in probs:
            p = p.tolist()
            score = p[2] - p[0]   # positive - negative
            scores.append(score)

        # Final average sentiment
        final_score = sum(scores) / len(scores)

        return float(final_score)

    except Exception as e:
        print("Error:", e)
        return 0.0  # fallback to neutral

🔹 STEP 4: Test it

In [ ]:
print(get_sentiment_score("AAPL"))

0.02736200185492635


🔥 STEP 5: Add FastAPI (paste below your code)

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

🔹 Add API code

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/sentiment/{symbol}")
def sentiment(symbol: str):
    score = get_sentiment_score(symbol)

    return {
        "stock": symbol,
        "sentiment_score": score
    }

🔥 STEP 6: Run server

In [ ]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run server in background
thread = Thread(target=run)
thread.start()

INFO:     Started server process [7067]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


STEP 3: Test inside Colab FIRST

In [ ]:
import requests

print(requests.get("http://127.0.0.1:8000/sentiment/AAPL").json())

INFO:     127.0.0.1:35612 - "GET /sentiment/AAPL HTTP/1.1" 200 OK
{'stock': 'AAPL', 'sentiment_score': 0.02736200185492635}


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("38hp5GZuLifwMVsTMr64mXup4vp_T1hcm3CgXqFtTE2CD2bz")

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://sidereal-benjamin-hurly.ngrok-free.dev" -> "http://localhost:8000"
